# EDA - Global Tech Job Market & Career Trajectories (2019-2026)

This notebook goes through the dataset step by step, following the process from the task sheet:
structure, missing values, duplicates, categorical consistency, logical checks, distributions,
relationships, time trends, feature engineering, and a summary of findings.

I'm keeping the code simple and adding notes as I go so I can explain each part later.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('tech_job_market_2019_2026.csv')
df.shape

In [ ]:
df.head()

## Step 1: Structural checks

First I just want to see how many rows/columns there are and what type pandas thinks each
column is. This matters because if a column that should be a number is actually stored as
text, things like `.mean()` won't work properly on it.

In [ ]:
df.info()

Looking at this:
- `posting_date`, `application_deadline`, `date_position_filled` are text, not real dates.
- `salary_min_usd` and `salary_max_usd` are text too, even though they are money amounts.
  That's because some values have a `$` sign, some have commas, some say "Not Disclosed".
- `visa_sponsorship_offered` and `layoff_within_12_months_flag` look like yes/no columns but
  they are stored as text and have more than 2 different spellings, as I found below.

## Step 2: Missing values

`isnull()` only finds actual blank/NaN cells. But some columns use a placeholder word like
"Unknown" or "TBD" instead of leaving the cell empty, so I also check for those separately.

In [ ]:
df.isnull().sum()

In [ ]:
# checking a few text columns for placeholder words that mean "missing" but aren't NaN
print(df['primary_programming_language'].value_counts().tail(5))
print(df['salary_min_usd'].value_counts().head(5))

`primary_programming_language` has values like `Unknown` and `TBD` which are really missing
data, just written as text instead of being blank. Same thing with `salary_min_usd` having
`Not Disclosed` values. I'll fix these later when cleaning the columns.

I also want to check if `job_location_city` being missing has a pattern, since it's missing
for about 22% of rows.

In [ ]:
df[df['job_location_city'].isnull()]['work_mode'].value_counts()

Most of the missing cities are for remote jobs. That makes sense - if a job is remote, maybe
there isn't really a "city" for it. So this missing data isn't really an error, it's just how
remote jobs work. I'm not going to try to fill these in.

Same idea for `date_position_filled` - I'd guess it's missing when the job was never filled.

In [ ]:
df[df['date_position_filled'].isnull()]['position_status'].value_counts()

Yes - it's missing for jobs that are Open, Cancelled or Withdrawn, which makes sense since
those jobs were never actually filled. So this is expected missingness, not bad data.

## Step 3: Duplicates

`job_id` should be unique for each posting, so I want to check if any id shows up more than
once, and also check for fully duplicate rows.

In [ ]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Repeated job_ids:", df['job_id'].duplicated().sum())

There are more repeated job_ids (2000) than duplicate rows (1193). That means some repeated
ids are actually the exact same row twice, but some repeated ids belong to completely
different job postings that just happen to share the same id by mistake. Let me check one of
each to be sure.

In [ ]:
# pick one repeated id and look at it
some_id = df[df['job_id'].duplicated(keep=False)]['job_id'].iloc[0]
df[df['job_id'] == some_id][['job_id', 'company_name', 'posting_date', 'position_status']]

For real duplicate rows, I'll just drop them since they don't add any new information. For
job_ids that are reused for different postings, I won't delete anything (that would lose real
data) - I'll just keep this in mind as a data quality issue: `job_id` isn't 100% reliable as a
unique key in this dataset.

In [ ]:
print("company names with extra spaces:", df['company_name'].str.strip().ne(df['company_name']).sum())

## Step 4: Making categories consistent

Now I build a cleaned copy of the data (`df2`) so I still have the original `df` to compare
against. I'll go column by column and fix things I noticed.

In [ ]:
df2 = df.copy()

### work_mode

Let's see all the different spellings.

In [ ]:
df['work_mode'].value_counts()

There are really only 3 categories here (Remote, Hybrid, Onsite) but they're spelled in
different ways (capital letters, "WFH", "On-site" with a dash, etc). I'll write a small
function to map all of these to one clean value.

In [ ]:
def clean_work_mode(value):
    value = str(value).strip().lower()
    if value in ['remote', 'work from home', 'wfh']:
        return 'Remote'
    elif value == 'hybrid':
        return 'Hybrid'
    elif value in ['onsite', 'on-site', 'in-office']:
        return 'Onsite'
    else:
        return np.nan

df2['work_mode_clean'] = df2['work_mode'].apply(clean_work_mode)
df2['work_mode_clean'].value_counts()

### visa_sponsorship_offered and layoff_within_12_months_flag

These are supposed to be yes/no columns but have several different spellings for yes and no.

In [ ]:
print(df['visa_sponsorship_offered'].value_counts())
print(df['layoff_within_12_months_flag'].value_counts())

In [ ]:
def clean_yes_no(value):
    value = str(value).strip().lower()
    if value in ['y', 'yes', 'true', '1']:
        return True
    elif value in ['n', 'no', 'false', '0']:
        return False
    else:
        return np.nan

df2['visa_sponsorship_clean'] = df2['visa_sponsorship_offered'].apply(clean_yes_no)
df2['layoff_flag_clean'] = df2['layoff_within_12_months_flag'].apply(clean_yes_no)

### sector

Let's check this one too.

In [ ]:
df['sector'].value_counts()

Same category, different capitalization, like `cloud/infra` and `Cloud/Infra`. I'll use
`.str.title()` to fix most of them, but a few words like "HealthTech" and "SaaS" get messed up
by `.title()` because it capitalizes every word, so I fix those by hand afterward.

In [ ]:
df2['sector'] = df2['sector'].str.strip().str.title()

fix_these = {
    'Ai/Ml': 'AI/ML',
    'Saas': 'SaaS',
    'E-Commerce': 'E-commerce',
    'Social/Adtech': 'Social/AdTech',
    'Edtech': 'EdTech',
    'Healthtech': 'HealthTech',
}
df2['sector'] = df2['sector'].replace(fix_these)
df2['sector'].value_counts()

In [ ]:
# also clean up extra spaces in company names
df2['company_name'] = df2['company_name'].str.strip()

### salary columns

These have `$` signs, commas, the word "USD", "Not Disclosed", and even a `-1` value which I
think is a placeholder for "no salary given" rather than a real salary. I'll write a function
to clean these up and turn them into normal numbers.

In [ ]:
def clean_salary(value):
    if pd.isnull(value):
        return np.nan
    value = str(value).strip()
    if value.lower() in ['not disclosed', 'n/a', 'na', 'unknown', '']:
        return np.nan
    value = value.replace('$', '').replace('USD', '').replace(',', '').strip()
    try:
        number = float(value)
    except ValueError:
        return np.nan
    if number < 0:
        return np.nan
    return number

df2['salary_min_clean'] = df2['salary_min_usd'].apply(clean_salary)
df2['salary_max_clean'] = df2['salary_max_usd'].apply(clean_salary)
df2[['salary_min_clean', 'salary_max_clean']].describe()

In [ ]:
# also fix the "Unknown"/"TBD" placeholders in programming language
df2['primary_programming_language'] = df2['primary_programming_language'].replace(
    {'Unknown': np.nan, 'TBD': np.nan}
)

In [ ]:
# now drop the real duplicate rows (this only removes rows that are 100% identical)
print("rows before:", len(df2))
df2 = df2.drop_duplicates()
print("rows after:", len(df2))

## Step 5: Logical checks

Now I check if the numbers actually make sense compared to each other. For example, the
minimum years of experience shouldn't be higher than the maximum.

In [ ]:
exp_problem = df2['years_experience_min'] > df2['years_experience_max']
print("min experience bigger than max experience:", exp_problem.sum())

offer_problem1 = df2['offers_accepted'] > df2['offers_extended']
print("more offers accepted than offers extended:", offer_problem1.sum())

offer_problem2 = df2['offers_extended'] > df2['number_of_applicants']
print("more offers extended than applicants:", offer_problem2.sum())

negative_applicants = df2['number_of_applicants'] < 0
print("negative number of applicants:", negative_applicants.sum())

In [ ]:
# parse the date columns properly so I can compare them
df2['posting_date'] = pd.to_datetime(df2['posting_date'], errors='coerce')
df2['application_deadline'] = pd.to_datetime(df2['application_deadline'], errors='coerce')
df2['date_position_filled'] = pd.to_datetime(df2['date_position_filled'], errors='coerce')

date_problem = df2['posting_date'] > df2['date_position_filled']
print("posted after it was already filled (impossible):", date_problem.sum())

All of these problems together are only about 1.5% of the rows, so I don't want to just
delete all of them - that could throw away good data in the other columns. Instead I'll add
one flag column that marks a row as having a logic problem, and I'll just be careful not to
use those specific numbers (like experience or dates) for rows that are flagged.

In [ ]:
df2['has_logic_issue'] = exp_problem | offer_problem1 | offer_problem2 | negative_applicants | date_problem.fillna(False)
df2['has_logic_issue'].sum()

## Step 6: Looking at distributions and outliers

Now that salaries are actual numbers and dates are actual dates, I can plot them and check
for weird extreme values.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df2['company_size_employees'], bins=40)
plt.title('Company size (number of employees)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df2['salary_min_clean'].dropna(), bins=40)
plt.title('Minimum salary (USD)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df2['number_of_applicants'])
plt.title('Number of applicants (boxplot to see outliers)')
plt.show()

Company size is spread out a lot, from very small startups to companies with almost 500,000
employees, which makes sense given the different company types in this dataset.

The number of applicants has some very high outliers (one posting has over 36,000 applicants)
and also some negative numbers, which isn't possible in real life, so those negative ones are
already counted as a data problem above.

## Step 7: Comparing variables (univariate, bivariate, multivariate)

### One variable at a time

In [ ]:
plt.figure(figsize=(8, 4))
df2['sector'].value_counts().plot(kind='bar')
plt.title('Number of postings per sector')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
order = ['Intern', 'Entry', 'Mid', 'Senior', 'Lead', 'Principal']
df2['seniority_level'].value_counts().reindex(order).plot(kind='bar')
plt.title('Number of postings per seniority level')
plt.show()

### Two variables at a time

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='seniority_level', y='salary_min_clean', order=order)
plt.title('Salary by seniority level')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df2, x='work_mode_clean', y='salary_min_clean')
plt.title('Salary by work mode')
plt.show()

### A few variables together

In [ ]:
numeric_columns = ['years_experience_min', 'years_experience_max', 'number_of_applicants',
                    'number_of_interview_rounds', 'offers_extended', 'offers_accepted',
                    'salary_min_clean', 'salary_max_clean', 'company_size_employees']

plt.figure(figsize=(8, 6))
sns.heatmap(df2[numeric_columns].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation between numeric columns')
plt.show()

*(Write my own notes here after looking at the actual charts - does salary really go up with
seniority level, is there a real difference between work modes, which numeric columns move
together in the heatmap?)*

## Step 8: Looking at trends over time

In [ ]:
postings_per_month = df2.set_index('posting_date').resample('ME').size()

plt.figure(figsize=(12, 4))
postings_per_month.plot()
plt.title('Number of postings per month, 2019-2026')
plt.ylabel('Postings')
plt.show()

*(Notes on what the chart actually shows - does the number of postings go up over time, is
there a dip anywhere, does it look seasonal? Since this is a synthetic dataset, I'll describe
what the chart shows rather than assuming it matches anything about the real job market.)*

## Step 9: Creating new columns (feature engineering)

The task asks for a few specific new columns, so I'll build those here.

**time_to_fill_days** - how many days between the job being posted and being filled. This can
only be calculated for jobs that were actually filled, and I make sure not to use rows with a
date problem from Step 5.

In [ ]:
df2['time_to_fill_days'] = (df2['date_position_filled'] - df2['posting_date']).dt.days
df2.loc[df2['time_to_fill_days'] < 0, 'time_to_fill_days'] = np.nan
df2['time_to_fill_days'].describe()

**offer_accept_ratio** - what fraction of the offers made were accepted. I set this to
missing (not zero) when no offers were extended at all, since dividing by zero doesn't make
sense.

In [ ]:
df2['offer_accept_ratio'] = df2['offers_accepted'] / df2['offers_extended']
df2.loc[df2['offers_extended'] == 0, 'offer_accept_ratio'] = np.nan
df2['offer_accept_ratio'].describe()

**avg_salary_usd** - the middle point of the min and max salary, using the already-cleaned
salary columns.

In [ ]:
df2['avg_salary_usd'] = (df2['salary_min_clean'] + df2['salary_max_clean']) / 2
df2['avg_salary_usd'].describe()

**required_tech_stack** - right now this is just one long text string per row, like
`"Python, SQL, AWS"`. I'll split it into a list and also count how many technologies are
listed, then find the most common individual technologies across the whole dataset.

In [ ]:
df2['tech_stack_list'] = df2['required_tech_stack'].str.split(', ')
df2['tech_stack_count'] = df2['tech_stack_list'].apply(len)

all_tech = df2['tech_stack_list'].explode()
top_tech = all_tech.value_counts().head(10)

plt.figure(figsize=(8, 5))
top_tech.plot(kind='barh')
plt.title('Most requested technologies (top 10)')
plt.gca().invert_yaxis()
plt.show()

## Step 10: Summary of data quality issues found

| Issue | How I found it | How many | What I did |
|---|---|---|---|
| Dates stored as text | `df.info()` | 3 columns | Converted with `pd.to_datetime()` |
| Salary stored as text ($, commas, "USD") | Looking at values | ~111,682 non-missing values | Cleaned into numbers |
| `-1` in salary columns | Checking for negative numbers | 737 | Treated as missing |
| "Not Disclosed" in salary columns | Looking at values | 686 | Treated as missing |
| "Unknown"/"TBD" in programming language | Looking at values | 1,093 | Treated as missing |
| Missing city | `isnull()` + checking work_mode | 25,107 rows | Left as missing - mostly remote jobs |
| Missing date_position_filled | `isnull()` + checking position_status | 25,781 rows | Left as missing - job wasn't filled |
| work_mode spelled 13 different ways | `.value_counts()` | ~5,600 rows | Mapped to 3 clean categories |
| visa/layoff columns with mixed yes/no spelling | `.value_counts()` | all rows | Mapped to True/False |
| sector spelled with different capitalization | `.value_counts()` | 24 raw values, really 12 | Fixed with `.title()` + manual fixes |
| Extra spaces in company name | `.str.strip()` check | 1,694 rows | Stripped |
| Fully duplicate rows | `df.duplicated()` | 1,193 rows | Dropped |
| Same job_id used for different postings | Checking rows with same id | 807 ids | Kept, but noted as a data quality issue |
| min experience > max experience | Comparing the 2 columns | 315 rows | Flagged, not deleted |
| offers accepted > offers extended | Comparing the 2 columns | 597 rows | Flagged, not deleted |
| offers extended > applicants | Comparing the 2 columns | 374 rows | Flagged, not deleted |
| Negative applicants | Checking for negative numbers | 247 rows | Flagged, not deleted |
| Posted after it was filled | Comparing dates | 445 rows | Flagged, excluded from time_to_fill_days |

### Notes / assumptions
- This is a synthetic dataset, so any pattern I see in the time trend is just describing this
  file, not making a claim about the real job market.
- I didn't fill in (impute) any missing values - I left them as missing everywhere, since
  guessing a salary or applicant count would just be making up data.
- The rows with logic problems (about 1.5% of the data) are still in the dataset, I just didn't
  use their broken values for calculations that depend on them.